In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
from torchvision.models import VGG16_Weights

In [ ]:
from ssd import SSD
from utils import normalised_gt_coords,corner_to_center,iou,center_to_corner,decode,encode ,matching



"""targets (tensor): Ground truth boxes and labels for a batch,
                shape: [batch_size,num_objs,5] (last idx is the label)."""
                
gt_boxes_batch = [
    # Image 1
    [
        (75, 80, 60, 90,1),
        (220, 70, 50, 40,2),
        (150, 170, 140, 110,1),
        (260, 240, 70, 120,1),
        (45, 230, 80, 60,2)
    ],

    # Image 2
    [
        (30, 40, 50, 60,1),
        (200, 120, 80, 90,3),
        (180, 160, 100, 110,3),
        (250, 210, 60, 80,3),
        (90, 200, 70, 50,4)
    ],

    # Add 8 more images...
]

# Convert to tensor
gtboxes = torch.tensor(gt_boxes_batch, dtype=torch.float32)
gtboxes.shape


# gtboxes=torch.tensor(gt_boxes, dtype=torch.float32)
# gt=center_to_corner(normalised_gt_coords(gtboxes,300,300))
# labels=torch.tensor([1,2,3,1,2]).unsqueeze(1)
# labels

torch.Size([2, 5, 5])

In [3]:
vgg = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1).features
vgg[16] = nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True)
base=nn.ModuleList(vgg[:30])#until 5_3 layer
base




ModuleList(
  (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU(inplace=True)
  (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (3): ReLU(inplace=True)
  (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (6): ReLU(inplace=True)
  (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (8): ReLU(inplace=True)
  (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (11): ReLU(inplace=True)
  (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (13): ReLU(inplace=True)
  (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (15): ReLU(inplace=True)
  (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=True)
  (17): Conv2d(256, 512, kernel_siz

In [4]:
X=torch.randn((10,3,300,300))

model=SSD(base,21)

In [5]:
model.anchors

tensor([[0.0000, 0.0000, 0.1435, 0.1435],
        [0.0000, 0.0000, 0.1699, 0.1435],
        [0.0000, 0.0000, 0.1962, 0.1435],
        ...,
        [0.0500, 0.0500, 0.9500, 0.9500],
        [0.0000, 0.1818, 1.0000, 0.8182],
        [0.1818, 0.0000, 0.8182, 1.0000]])

In [6]:
regressions,classifications,layers_for_prediction=model.forward(X)

In [9]:
for_anchor_coords_encoded,for_anchor_classes,positives_mask=matching(model.anchors,gt,labels,variances=[0.1, 0.2])

In [11]:
"""#sa,ity chekc 
from utils import decode
decode(for_anchor_coords_encoded,model.anchors,[0.1,0.2])[312]


#shoud show tensor([0.1500, 0.1167, 0.3500, 0.4167])"""

'#sa,ity chekc \nfrom utils import decode\ndecode(for_anchor_coords_encoded,model.anchors,[0.1,0.2])[312]\n\n\n#shoud show tensor([0.1500, 0.1167, 0.3500, 0.4167])'

In [13]:
regressions.shape

torch.Size([10, 8732, 4])

In [14]:
for_anchor_coords_encoded.shape

torch.Size([8732, 4])

In [12]:
def smoothL1(x):
    return torch.where(torch.abs(x)<1,0.5*x**2,torch.abs(x)-0.5)
    
#lloc loss over positives only 
smoothL1(regressions-for_anchor_coords_encoded)[positives_mask].sum()

IndexError: The shape of the mask [8732] at index 0 does not match the shape of the indexed tensor [10, 8732, 4] at index 0

In [36]:
for_anchor_coords.shape

torch.Size([8732, 4])

# model forward pass 

In [44]:
X=torch.randn((10,3,300,300))

model=SSD(base,21)
regressions,classifications,layers_for_prediction=model.forward(X)

In [45]:
classifications.shape

torch.Size([10, 8732, 21])

In [46]:
regressions.shape



torch.Size([10, 8732, 4])

In [30]:
for el  in layers_for_prediction:
    print(el.shape[2:])



torch.Size([38, 38])
torch.Size([19, 19])
torch.Size([10, 10])
torch.Size([5, 5])
torch.Size([3, 3])
torch.Size([1, 1])
